<a href="https://www.kaggle.com/code/robiulhasanjisan/toyota-stock-eda-ml?scriptVersionId=308162269" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install arch

In [ ]:


# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Utilities
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Statistical analysis
from scipy import stats
from scipy.stats import jarque_bera, shapiro, normaltest
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Advanced models
from arch import arch_model

# Visualization (interactive)
import plotly.graph_objs as go
import plotly.express as px
from plotly.subplots import make_subplots

# ML / preprocessing
from sklearn.preprocessing import StandardScaler

# Feature analysis
from statsmodels.stats.outliers_influence import variance_inflation_factor
from pandas.plotting import scatter_matrix

# Notebook display
from IPython.display import display


# Display & Style Configuration


# Pandas display
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.expand_frame_repr', False)
pd.set_option('display.precision', 2)

# Plot styles
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

In [ ]:

from IPython.display import display

# --- Styling function ---
def style_df(df):
    return df.style \
        .background_gradient(cmap='coolwarm') \
        .set_properties(**{
            'border': '1px solid #ddd',
            'padding': '6px',
            'text-align': 'center'
        }) \
        .set_table_styles([
            {
                'selector': 'th',
                'props': [
                    ('background', 'linear-gradient(90deg, #4b6cb7, #182848)'),
                    ('color', 'white'),
                    ('font-weight', 'bold'),
                    ('border', '1px solid white')
                ]
            }
        ])


In [ ]:

# --- Load the dataset ---
file_path = '/kaggle/input/datasets/mdmahfuzsumon/toyota-stock-market-intelligence-19802026/toyota_stock_v2_features.csv'
df = pd.read_csv(file_path, parse_dates=['Date'])
df.set_index('Date', inplace=True)
df.sort_index(inplace=True)

# Basic data info

In [ ]:
print(f" Dataset Shape       : {df.shape}")
print(f" Date Range          : {df.index.min()} to {df.index.max()}")
print(f" Total Trading Days  : {len(df):,}")
print(f" Features Available  : {list(df.columns)}\n")


In [ ]:

missing_total = df.isnull().sum().sum()
duplicates = df.duplicated().sum()
memory = df.memory_usage(deep=True).sum() / 1024**2
print(" Data Quality Report:")
print(f"   - Missing Values    : {missing_total}")
print(f"   - Duplicate Rows    : {duplicates}")
print(f"   - Memory Usage      : {memory:.2f} MB\n")


In [ ]:


display(style_df(df.head()))

In [ ]:

display(style_df(df.tail()))

In [ ]:



df['Log_Return'] = np.log(df['Close'] / df['Close'].shift(1)).fillna(0)


# EDA

## Toyota Stock Price Evolution with Key Events and Recessions

In [ ]:

import matplotlib.pyplot as plt
import pandas as pd

fig, ax = plt.subplots(figsize=(16, 8))

# --- Plot Closing Price ---
ax.plot(df.index, df['Close'], color='#1f77b4', linewidth=2, alpha=0.9, label='Closing Price')

# --- Highlight Key Historical Events ---
events = {
    '1987-10-19': 'Black Monday',
    '1997-07-02': 'Asian Financial Crisis',
    '2000-03-10': 'Dot-com Bubble Burst',
    '2008-09-15': 'Global Financial Crisis',
    '2011-03-11': 'Japan Tsunami',
    '2020-03-11': 'COVID-19 Pandemic',
    '2022-02-24': 'Russia-Ukraine War'
}

event_colors = ['#d62728', '#ff7f0e', '#9467bd', '#d62728', '#ff7f0e', '#2ca02c', '#7f7f7f']

for idx, (date, label) in enumerate(events.items()):
    event_date = pd.to_datetime(date)
    if event_date in df.index:
        ax.axvline(event_date, color=event_colors[idx % len(event_colors)], linestyle='--', linewidth=1.8, alpha=0.7)
        ax.text(event_date, df['Close'].max()*0.92, label, rotation=90, fontsize=9, fontweight='bold',
                verticalalignment='top', horizontalalignment='right', color=event_colors[idx % len(event_colors)])

# --- Shade Major Recessions ---
recessions = [
    ('1990-07-01', '1991-03-01', 'Early 90s Recession'),
    ('2001-03-01', '2001-11-01', 'Dot-com Recession'),
    ('2007-12-01', '2009-06-01', 'Great Recession'),
    ('2020-02-01', '2020-04-01', 'COVID Recession')
]

for start, end, label in recessions:
    start_date = pd.to_datetime(start)
    end_date = pd.to_datetime(end)
    ax.axvspan(start_date, end_date, color='gray', alpha=0.2)
    ax.text(start_date + (end_date-start_date)/2, df['Close'].min()*1.05, label, 
            fontsize=9, fontstyle='italic', color='gray', ha='center')

# --- Titles and Labels ---
ax.set_title('Toyota Stock Price Evolution (1980-2026)', fontsize=18, fontweight='bold', pad=20)
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Price (USD)', fontsize=12)
ax.grid(True, alpha=0.3)
ax.legend(loc='upper left', fontsize=10)
ax.set_yscale('log')

# --- Annotate Total Growth ---
first_price = df['Close'].iloc[0]
last_price = df['Close'].iloc[-1]
growth = ((last_price - first_price) / first_price) * 100

ax.annotate(
    f'Total Growth: {growth:.1f}%\n${first_price:.2f} → ${last_price:.2f}',
    xy=(0.02, 0.95), xycoords='axes fraction',
    fontsize=11, fontweight='bold',
    bbox=dict(boxstyle="round,pad=0.3", facecolor="yellow", alpha=0.7)
)

plt.tight_layout()
plt.show()

# --- Print Summary Statistics ---
print(f"\n Toyota Stock Price Statistics:")
print(f"   Minimum Price: ${df['Close'].min():.2f} ({df['Close'].idxmin().strftime('%Y-%m-%d')})")
print(f"   Maximum Price: ${df['Close'].max():.2f} ({df['Close'].idxmax().strftime('%Y-%m-%d')})")
print(f"   Average Price: ${df['Close'].mean():.2f}")
print(f"   Price Growth: {growth:.1f}% over {len(df)/252:.1f} years")

## Distribution Analysis

In [ ]:


fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# --- 1. Daily Returns Histogram ---
ax1 = axes[0, 0]
returns = df['Return'].dropna()
ax1.hist(returns, bins=100, density=True, alpha=0.7, color='#1f77b4', edgecolor='black')
mu, sigma = stats.norm.fit(returns)
x = np.linspace(returns.min(), returns.max(), 100)
ax1.plot(x, stats.norm.pdf(x, mu, sigma), 'r-', linewidth=2, label=f'Normal Fit (μ={mu:.4f}, σ={sigma:.4f})')
ax1.set_title('Daily Returns Distribution', fontsize=13, fontweight='bold')
ax1.set_xlabel('Returns')
ax1.set_ylabel('Density')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Statistics box
jb_stat, jb_pvalue = jarque_bera(returns)
stats_text = f"Skewness: {returns.skew():.3f}\nKurtosis: {returns.kurtosis():.3f}\nJarque-Bera: {jb_stat:.2f}\nP-value: {jb_pvalue:.4f}"
ax1.text(0.95, 0.95, stats_text, transform=ax1.transAxes, verticalalignment='top',
         horizontalalignment='right', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# --- 2. Log Returns Histogram ---
ax2 = axes[0, 1]
log_returns = df['Log_Return'].dropna()
ax2.hist(log_returns, bins=100, density=True, alpha=0.7, color='#2ca02c', edgecolor='black')
mu_log, sigma_log = stats.norm.fit(log_returns)
x_log = np.linspace(log_returns.min(), log_returns.max(), 100)
ax2.plot(x_log, stats.norm.pdf(x_log, mu_log, sigma_log), 'orange', linewidth=2)
ax2.set_title('Log Returns Distribution', fontsize=13, fontweight='bold')
ax2.set_xlabel('Log Returns')
ax2.set_ylabel('Density')
ax2.grid(True, alpha=0.3)

# --- 3. Q-Q Plot ---
ax3 = axes[0, 2]
stats.probplot(returns, dist="norm", plot=ax3)
ax3.set_title('Q-Q Plot (Normality Check)', fontsize=13, fontweight='bold')
ax3.grid(True, alpha=0.3)

# --- 4. Boxplot by Decade ---
ax4 = axes[1, 0]
df['Decade'] = (df.index.year // 10) * 10
returns_by_decade = [df[df['Decade']==dec]['Return'].dropna() for dec in sorted(df['Decade'].unique())]
bp = ax4.boxplot(returns_by_decade, labels=[f'{int(d)}s' for d in sorted(df['Decade'].unique())], patch_artist=True)
colors = ['#ff9999','#66b3ff','#99ff99','#ffcc99','#c2c2f0','#ffb3e6']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
ax4.set_title('Returns by Decade', fontsize=13, fontweight='bold')
ax4.set_xlabel('Decade')
ax4.set_ylabel('Returns')
ax4.grid(True, alpha=0.3)
ax4.axhline(0, color='red', linestyle='--', alpha=0.5)

# --- 5. Monthly Returns Heatmap ---
ax5 = axes[1, 1]
pivot_returns = df.pivot_table(values='Return', index=df.index.year, columns=df.index.month, aggfunc='mean')
im = ax5.imshow(pivot_returns, cmap='coolwarm', aspect='auto', vmin=-0.02, vmax=0.02)
ax5.set_title('Average Monthly Returns Heatmap', fontsize=13, fontweight='bold')
ax5.set_xlabel('Month')
ax5.set_ylabel('Year')
ax5.set_xticks(range(12))
ax5.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
plt.colorbar(im, ax=ax5, label='Average Return')

# --- 6. Cumulative Returns ---
ax6 = axes[1, 2]
df['Cumulative_Return'] = (1 + df['Return']).cumprod()
ax6.plot(df.index, df['Cumulative_Return'], color='#9467bd', linewidth=2)
ax6.set_title('Cumulative Returns (1980-2026)', fontsize=13, fontweight='bold')
ax6.set_xlabel('Date')
ax6.set_ylabel('Cumulative Return (Log Scale)')
ax6.set_yscale('log')
ax6.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()



In [ ]:
# --- Returns Summary Statistics ---
print("\n\033[1;34m Returns Statistical Summary\033[0m\n")  

# ANSI colors
GREEN = '\033[92m'
RED = '\033[91m'
YELLOW = '\033[93m'
CYAN = '\033[96m'
BOLD = '\033[1m'
RESET = '\033[0m'

# Basic stats
mean_daily = returns.mean()
std_daily = returns.std()
annualized_mean = mean_daily * 252
annualized_std = std_daily * np.sqrt(252)
sharpe_ratio = mean_daily / std_daily * np.sqrt(252)
max_gain = returns.max()
max_loss = returns.min()
max_gain_date = returns.idxmax().strftime('%Y-%m-%d')
max_loss_date = returns.idxmin().strftime('%Y-%m-%d')
positive_days = (returns > 0).sum()
negative_days = (returns < 0).sum()
positive_pct = (returns > 0).mean() * 100
negative_pct = (returns < 0).mean() * 100

# Print formatted summary
print(f"{CYAN}{'Mean Daily Return':<35}:{RESET} {YELLOW}{mean_daily:.6f}{RESET} "
      f"({YELLOW}{annualized_mean*100:.2f}% annualized{RESET})")
print(f"{CYAN}{'Std Daily Return':<35}:{RESET} {YELLOW}{std_daily:.6f}{RESET} "
      f"({YELLOW}{annualized_std*100:.2f}% annualized{RESET})")
print(f"{CYAN}{'Sharpe Ratio (annualized)':<35}:{RESET} {GREEN}{sharpe_ratio:.3f}{RESET}")
print(f"{CYAN}{'Maximum Daily Gain':<35}:{RESET} {GREEN}{max_gain:.4f}{RESET} ({max_gain_date})")
print(f"{CYAN}{'Maximum Daily Loss':<35}:{RESET} {RED}{max_loss:.4f}{RESET} ({max_loss_date})")
print(f"{CYAN}{'Positive Days':<35}:{RESET} {GREEN}{positive_days:,}{RESET} ({GREEN}{positive_pct:.1f}%{RESET})")
print(f"{CYAN}{'Negative Days':<35}:{RESET} {RED}{negative_days:,}{RESET} ({RED}{negative_pct:.1f}%{RESET})")

## Volume & Liquidity Analysis

In [ ]:


fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# --- 1. Volume over Time ---
ax1 = axes[0, 0]
ax1.fill_between(df.index, df['Volume'], alpha=0.2, color='#2ca02c')
ax1.plot(df.index, df['Volume'], linewidth=0.7, color='#1f7a1f', alpha=0.8, label='Daily Volume')

# Volume moving averages
df['Volume_MA_20'] = df['Volume'].rolling(20).mean()
df['Volume_MA_100'] = df['Volume'].rolling(100).mean()
ax1.plot(df.index, df['Volume_MA_20'], color='#ff7f0e', label='20-day MA', linewidth=1.5)
ax1.plot(df.index, df['Volume_MA_100'], color='#d62728', label='100-day MA', linewidth=1.5)
ax1.set_title('Trading Volume Over Time', fontsize=13, fontweight='bold')
ax1.set_xlabel('Date')
ax1.set_ylabel('Volume (Log Scale)')
ax1.set_yscale('log')
ax1.grid(True, alpha=0.3)
ax1.legend()

# --- 2. Volume Distribution ---
ax2 = axes[0, 1]
ax2.hist(df['Volume'].dropna(), bins=100, alpha=0.7, color='#1f77b4', edgecolor='black', log=True)
ax2.set_title('Volume Distribution (Log Scale)', fontsize=13, fontweight='bold')
ax2.set_xlabel('Volume')
ax2.set_ylabel('Frequency (Log)')
ax2.set_xscale('log')
ax2.grid(True, alpha=0.3)

# Stats box
vol_stats = f"Median: {df['Volume'].median():,.0f}\nMean: {df['Volume'].mean():,.0f}\nMax: {df['Volume'].max():,.0f}"
ax2.text(0.95, 0.95, vol_stats, transform=ax2.transAxes,
         verticalalignment='top', horizontalalignment='right',
         bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# --- 3. Price-Volume Relationship ---
ax3 = axes[1, 0]
scatter = ax3.scatter(df['Close'], df['Volume'], c=df.index.year, cmap='viridis', s=10, alpha=0.6)
ax3.set_title('Price-Volume Relationship', fontsize=13, fontweight='bold')
ax3.set_xlabel('Closing Price ($)')
ax3.set_ylabel('Volume')
ax3.set_yscale('log')
plt.colorbar(scatter, ax=ax3, label='Year')
ax3.grid(True, alpha=0.3)

# --- 4. Monthly Average Volume ---
ax4 = axes[1, 1]
monthly_volume = df.groupby(df.index.month)['Volume'].mean()
colors = plt.cm.viridis(np.linspace(0, 1, 12))
bars = ax4.bar(range(1, 13), monthly_volume.values, color=colors, edgecolor='black', alpha=0.8)
ax4.set_title('Average Monthly Trading Volume', fontsize=13, fontweight='bold')
ax4.set_xlabel('Month')
ax4.set_ylabel('Average Volume')
ax4.set_xticks(range(1, 13))
ax4.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
ax4.grid(True, alpha=0.3, axis='y')

# Add bar labels
for bar, vol in zip(bars, monthly_volume.values):
    ax4.text(bar.get_x() + bar.get_width()/2, vol, f'{vol/1e6:.1f}M', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
# --- Volume Analysis Summary ---
print("\n\033[1;34m Volume Analysis Summary\033[0m\n")  

# ANSI colors
GREEN = '\033[92m'
RED = '\033[91m'
YELLOW = '\033[93m'
CYAN = '\033[96m'
BOLD = '\033[1m'
RESET = '\033[0m'

# Basic volume stats
avg_vol = df['Volume'].mean()
med_vol = df['Volume'].median()
max_vol = df['Volume'].max()
min_vol = df['Volume'].min()
max_date = df['Volume'].idxmax().strftime('%Y-%m-%d')
min_date = df['Volume'].idxmin().strftime('%Y-%m-%d')

print(f"{CYAN}{'Average Daily Volume':<30}:{RESET} {YELLOW}{avg_vol:,.0f}{RESET}")
print(f"{CYAN}{'Median Daily Volume':<30}:{RESET} {YELLOW}{med_vol:,.0f}{RESET}")
print(f"{CYAN}{'Highest Volume Day':<30}:{RESET} {RED}{max_vol:,.0f}{RESET} ({max_date})")
print(f"{CYAN}{'Lowest Volume Day':<30}:{RESET} {RED}{min_vol:,.0f}{RESET} ({min_date})")

# Detect volume spikes
volume_threshold = avg_vol + 3*df['Volume'].std()
volume_spikes = df[df['Volume'] > volume_threshold]
print(f"\n {BOLD}Volume Spikes{RESET} (>{volume_threshold:,.0f}): {len(volume_spikes)} spikes")

# Top 5 spikes
top_spikes = volume_spikes.nlargest(5, 'Volume')
print(f"{BOLD}   Top 5 spikes:{RESET}")
for date, row in top_spikes.iterrows():
    print(f"      {date.strftime('%Y-%m-%d')}: {RED}{row['Volume']:,.0f}{RESET} "
          f"(Close: {GREEN}${row['Close']:.2f}{RESET})")

## Seasonality & Time Effects Analysis

In [ ]:


fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# --- 1. Average Monthly Returns ---
ax1 = axes[0, 0]
monthly_returns = df.groupby(df.index.month)['Return'].mean() * 100
colors = ['red' if x < 0 else 'green' for x in monthly_returns]
bars = ax1.bar(range(1, 13), monthly_returns.values, color=colors, alpha=0.7, edgecolor='black')
ax1.set_title('Average Monthly Returns (%)', fontsize=13, fontweight='bold')
ax1.set_xlabel('Month')
ax1.set_ylabel('Average Return (%)')
ax1.axhline(0, color='black', linestyle='-', linewidth=0.5)
ax1.set_xticks(range(1, 13))
ax1.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
ax1.grid(True, alpha=0.3, axis='y')

for bar, ret in zip(bars, monthly_returns.values):
    ha = 'center'
    va = 'bottom' if ret > 0 else 'top'
    ax1.text(bar.get_x() + bar.get_width()/2, ret, f'{ret:.2f}%', ha=ha, va=va, fontsize=8)

# --- 2. Day-of-Week Effect ---
ax2 = axes[0, 1]
dow_returns = df.groupby(df.index.dayofweek)['Return'].mean() * 100
dow_labels = ['Mon','Tue','Wed','Thu','Fri']
colors = ['red' if x < 0 else 'green' for x in dow_returns]
bars = ax2.bar(dow_labels, dow_returns.values, color=colors, alpha=0.7, edgecolor='black')
ax2.set_title('Average Returns by Day of Week (%)', fontsize=13, fontweight='bold')
ax2.set_xlabel('Day of Week')
ax2.set_ylabel('Average Return (%)')
ax2.axhline(0, color='black', linestyle='-', linewidth=0.5)
ax2.grid(True, alpha=0.3, axis='y')

# --- 3. Quarterly Returns ---
ax3 = axes[0, 2]
quarterly_returns = df.groupby(df.index.quarter)['Return'].mean() * 100
colors = ['lightblue', 'lightgreen', 'lightcoral', 'gold']
bars = ax3.bar(range(1, 5), quarterly_returns.values, color=colors, alpha=0.7, edgecolor='black')
ax3.set_title('Average Quarterly Returns (%)', fontsize=13, fontweight='bold')
ax3.set_xlabel('Quarter')
ax3.set_ylabel('Average Return (%)')
ax3.set_xticks(range(1, 5))
ax3.set_xticklabels(['Q1','Q2','Q3','Q4'])
ax3.axhline(0, color='black', linestyle='-', linewidth=0.5)
ax3.grid(True, alpha=0.3, axis='y')

# --- 4. Rolling 30-day Mean & Std ---
ax4 = axes[1, 0]
df['Rolling_Mean_30'] = df['Close'].rolling(30).mean()
df['Rolling_Std_30'] = df['Close'].rolling(30).std()
ax4.plot(df.index, df['Close'], label='Close Price', color='blue', linewidth=1, alpha=0.5)
ax4.plot(df.index, df['Rolling_Mean_30'], label='30-day MA', color='red', linewidth=2)
ax4.fill_between(df.index, df['Rolling_Mean_30'] - df['Rolling_Std_30'],
                 df['Rolling_Mean_30'] + df['Rolling_Std_30'], color='gray', alpha=0.3)
ax4.set_title('Rolling Mean & Std (30-day)', fontsize=13, fontweight='bold')
ax4.set_xlabel('Date')
ax4.set_ylabel('Price ($)')
ax4.legend()
ax4.grid(True, alpha=0.3)

# --- 5. Rolling Correlation: Price vs Volume ---
ax5 = axes[1, 1]
df['Rolling_Corr_Price_Volume'] = df['Close'].rolling(60).corr(df['Volume'])
ax5.plot(df.index, df['Rolling_Corr_Price_Volume'], color='purple', linewidth=1.5)
ax5.axhline(0, color='black', linestyle='-', linewidth=0.5)
ax5.set_title('Rolling Correlation: Price vs Volume (60-day)', fontsize=13, fontweight='bold')
ax5.set_xlabel('Date')
ax5.set_ylabel('Correlation')
ax5.grid(True, alpha=0.3)

# --- 6. Rolling Correlation: Returns vs Volatility ---
ax6 = axes[1, 2]
df['Rolling_Corr_Return_Vol'] = df['Return'].rolling(60).corr(df['Volatility_30'])
ax6.plot(df.index, df['Rolling_Corr_Return_Vol'], color='orange', linewidth=1.5)
ax6.axhline(0, color='black', linestyle='-', linewidth=0.5)
ax6.set_title('Rolling Correlation: Returns vs Volatility (60-day)', fontsize=13, fontweight='bold')
ax6.set_xlabel('Date')
ax6.set_ylabel('Correlation')
ax6.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# --- Seasonality Statistical Tests ---
print("\n\033[1;34m Seasonality Statistical Tests\033[0m\n")  # Bold Blue Title

from scipy.stats import f_oneway

# ANSI colors
GREEN = '\033[92m'
RED = '\033[91m'
YELLOW = '\033[93m'
CYAN = '\033[96m'
BOLD = '\033[1m'
RESET = '\033[0m'

# --- Month Effect ---
month_groups = [df[df.index.month == m]['Return'].dropna() for m in range(1, 13)]
f_stat, p_value = f_oneway(*month_groups)

print(f"{BOLD} Month Effect ANOVA{RESET}")
print(f"{CYAN}{'F-statistic':<25}:{RESET} {f_stat:.4f}")
print(f"{CYAN}{'P-value':<25}:{RESET} {p_value:.4f}")
significance = f"{GREEN} Significant{RESET}" if p_value < 0.05 else f"{RED} Not significant{RESET}"
print(f"{CYAN}{'Significance':<25}:{RESET} {significance}")

monthly_means = [grp.mean() for grp in month_groups]
best_month = monthly_means.index(max(monthly_means)) + 1
worst_month = monthly_means.index(min(monthly_means)) + 1
print(f"{CYAN}{'Highest avg return month':<25}:{RESET} {YELLOW}{best_month}{RESET}")
print(f"{CYAN}{'Lowest avg return month':<25}:{RESET} {YELLOW}{worst_month}{RESET}")

print("\n" + "-"*60 + "\n")

# --- Day-of-Week Effect ---
dow_groups = [df[df.index.dayofweek == d]['Return'].dropna() for d in range(5)]
f_stat, p_value = f_oneway(*dow_groups)

print(f"{BOLD} Day-of-Week Effect ANOVA{RESET}")
print(f"{CYAN}{'F-statistic':<25}:{RESET} {f_stat:.4f}")
print(f"{CYAN}{'P-value':<25}:{RESET} {p_value:.4f}")
significance = f"{GREEN} Significant{RESET}" if p_value < 0.05 else f"{RED} Not significant{RESET}"
print(f"{CYAN}{'Significance':<25}:{RESET} {significance}")

dow_names = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']
dow_means = [grp.mean() for grp in dow_groups]
best_day = dow_names[dow_means.index(max(dow_means))]
worst_day = dow_names[dow_means.index(min(dow_means))]
print(f"{CYAN}{'Highest avg return day':<25}:{RESET} {YELLOW}{best_day}{RESET}")
print(f"{CYAN}{'Lowest avg return day':<25}:{RESET} {YELLOW}{worst_day}{RESET}")

##  Correlation & Feature Relationships

In [ ]:



# --- 1. Prepare Correlation Matrix ---
correlation_features = ['Close', 'Volume', 'Return', 'RSI', 'MACD', 'MACD_Signal',
                        'MA_10', 'MA_50', 'Volatility_10', 'Volatility_30']

available_features = [f for f in correlation_features if f in df.columns]
corr_matrix = df[available_features].corr()

# --- 2. Correlation Heatmap ---
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='coolwarm', center=0, cbar_kws={'label': 'Correlation'})
plt.title('Feature Correlation Matrix', fontsize=14, fontweight='bold', pad=20)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# --- 3. Scatter Matrix for Key Features ---
scatter_features = ['Close', 'Volume', 'Return', 'RSI']
scatter_df = df[scatter_features].dropna().sample(min(5000, len(df)), random_state=42)

plt.figure(figsize=(12, 12))
scatter_matrix(scatter_df, alpha=0.3, diagonal='hist', figsize=(12,12))
plt.suptitle('Scatter Matrix of Key Features', fontsize=14, fontweight='bold', y=1.02)
plt.show()

# --- 4. Top Correlations with Close ---
print("\n Top Correlations with Close Price:")
close_corr = corr_matrix['Close'].sort_values(ascending=False)
for feature, corr in close_corr.head(10).items():
    if feature != 'Close':
        print(f"   {feature}: {corr:.4f}")

# --- 5. Multicollinearity Check (VIF) ---
X = df[available_features].dropna()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

vif_data = pd.DataFrame()
vif_data["Feature"] = available_features
vif_data["VIF"] = [variance_inflation_factor(X_scaled, i) for i in range(len(available_features))]
print("\n Variance Inflation Factor (VIF):")
print(vif_data.sort_values('VIF', ascending=False).to_string(index=False))
print("\n Note: VIF > 10 indicates high multicollinearity")

# --- 6. Correlations with Daily Returns ---
print("\n Correlations with Daily Returns:")
return_corr = df[available_features].corr()['Return'].sort_values(ascending=False)
for feature, corr in return_corr.head(10).items():
    print(f"   {feature}: {corr:.4f}")

## Volatility & Risk Analysis

In [ ]:

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1. Rolling volatility
ax1 = axes[0, 0]
df['Volatility_30'] = df['Return'].rolling(30).std() * np.sqrt(252) * 100  # Annualized %
df['Volatility_100'] = df['Return'].rolling(100).std() * np.sqrt(252) * 100

ax1.plot(df.index, df['Volatility_30'], linewidth=1.5, label='30-day Volatility', color='red')
ax1.plot(df.index, df['Volatility_100'], linewidth=1.5, label='100-day Volatility', color='orange', alpha=0.7)
ax1.fill_between(df.index, df['Volatility_30'], alpha=0.3, color='red')
ax1.set_title('Annualized Rolling Volatility', fontsize=12, fontweight='bold')
ax1.set_xlabel('Date')
ax1.set_ylabel('Volatility (%)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Highlight volatility spikes
vol_threshold = df['Volatility_30'].quantile(0.95)
high_vol_periods = df[df['Volatility_30'] > vol_threshold]
for date in high_vol_periods.index[::30]:  # Label every 30th spike to avoid clutter
    ax1.axvline(x=date, color='gray', linestyle='--', alpha=0.3, linewidth=0.5)

# 2. Volatility clustering visualization
ax2 = axes[0, 1]
ax2.plot(df.index, df['Return'] * 100, linewidth=0.5, alpha=0.5, color='blue', label='Daily Returns')
ax2.set_title('Volatility Clustering', fontsize=12, fontweight='bold')
ax2.set_xlabel('Date')
ax2.set_ylabel('Returns (%)')
ax2.grid(True, alpha=0.3)

# Highlight high volatility periods
for idx in high_vol_periods.index:
    ax2.axvspan(idx - pd.Timedelta(days=30), idx + pd.Timedelta(days=30), 
                alpha=0.1, color='red')

# 3. Volatility distribution
ax3 = axes[1, 0]
ax3.hist(df['Volatility_30'].dropna(), bins=50, alpha=0.7, color='red', edgecolor='black')
ax3.set_title('Volatility Distribution', fontsize=12, fontweight='bold')
ax3.set_xlabel('Annualized Volatility (%)')
ax3.set_ylabel('Frequency')
ax3.axvline(x=df['Volatility_30'].mean(), color='black', linestyle='--', 
            linewidth=2, label=f'Mean: {df["Volatility_30"].mean():.1f}%')
ax3.axvline(x=df['Volatility_30'].median(), color='blue', linestyle='--', 
            linewidth=2, label=f'Median: {df["Volatility_30"].median():.1f}%')
ax3.legend()
ax3.grid(True, alpha=0.3)

# 4. Risk metrics comparison
ax4 = axes[1, 1]
risk_metrics = {
    'VaR (95%)': df['Return'].quantile(0.05) * 100,
    'CVaR (95%)': df[df['Return'] <= df['Return'].quantile(0.05)]['Return'].mean() * 100,
    'Max Drawdown': (df['Close'] / df['Close'].cummax() - 1).min() * 100,
    'Downside Deviation': df[df['Return'] < 0]['Return'].std() * np.sqrt(252) * 100
}

colors = ['red', 'orange', 'darkred', 'coral']
bars = ax4.bar(risk_metrics.keys(), risk_metrics.values(), color=colors, alpha=0.7, edgecolor='black')
ax4.set_title('Risk Metrics', fontsize=12, fontweight='bold')
ax4.set_ylabel('Percentage (%)')
ax4.grid(True, alpha=0.3, axis='y')

# Add value labels
for bar, value in zip(bars, risk_metrics.values()):
    height = bar.get_height()
    ax4.text(bar.get_x() + bar.get_width()/2., height, f'{value:.2f}%',
             ha='center', va='bottom' if value > 0 else 'top', fontsize=9)

plt.tight_layout()
plt.show()

# GARCH Model Analysis
print("\n GARCH(1,1) Volatility Modeling:")
try:
    returns_100 = df['Return'].dropna() * 100
    model = arch_model(returns_100, vol='Garch', p=1, q=1, dist='normal')
    garch_result = model.fit(disp='off')
    
    print("\nGARCH(1,1) Model Results:")
    print(garch_result.summary())
    
    # Plot conditional volatility
    fig, ax = plt.subplots(figsize=(12, 5))
    conditional_vol = garch_result.conditional_volatility
    ax.plot(df.index[-len(conditional_vol):], conditional_vol, linewidth=1, color='purple')
    ax.set_title('GARCH(1,1) Conditional Volatility', fontsize=12, fontweight='bold')
    ax.set_xlabel('Date')
    ax.set_ylabel('Conditional Volatility (%)')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
except Exception as e:
    print(f"Could not fit GARCH model: {e}")

# Additional risk analysis
print("\n Advanced Risk Analysis:")
print(f"   Value at Risk (95%): {risk_metrics['VaR (95%)']:.2f}%")
print(f"   Conditional VaR (95%): {risk_metrics['CVaR (95%)']:.2f}%")
print(f"   Maximum Drawdown: {risk_metrics['Max Drawdown']:.2f}%")
print(f"   Downside Deviation (annualized): {risk_metrics['Downside Deviation']:.2f}%")
print(f"   Sortino Ratio: {df['Return'].mean()*252 / (risk_metrics['Downside Deviation']/100):.2f}")

## Technical Indicators Analysis

In [ ]:
 
fig, axes = plt.subplots(3, 1, figsize=(16, 14))

# Calculate Bollinger Bands if not already present
if 'BB_Upper' not in df.columns:
    df['MA_20'] = df['Close'].rolling(20).mean()
    df['BB_Upper'] = df['MA_20'] + (df['Close'].rolling(20).std() * 2)
    df['BB_Lower'] = df['MA_20'] - (df['Close'].rolling(20).std() * 2)

# 1. RSI with overbought/oversold levels
ax1 = axes[0]
ax1.plot(df.index, df['RSI'], linewidth=1.5, color='purple', label='RSI (14)')
ax1.axhline(y=70, color='red', linestyle='--', alpha=0.7, linewidth=1.5, label='Overbought (70)')
ax1.axhline(y=30, color='green', linestyle='--', alpha=0.7, linewidth=1.5, label='Oversold (30)')
ax1.fill_between(df.index, 70, df['RSI'], where=(df['RSI'] > 70), color='red', alpha=0.3)
ax1.fill_between(df.index, 30, df['RSI'], where=(df['RSI'] < 30), color='green', alpha=0.3)
ax1.set_title('Relative Strength Index (RSI)', fontsize=14, fontweight='bold')
ax1.set_ylabel('RSI')
ax1.set_ylim(0, 100)
ax1.legend(loc='upper left')
ax1.grid(True, alpha=0.3)

# Add RSI statistics
rsi_stats = f"Mean RSI: {df['RSI'].mean():.1f}\n"
rsi_stats += f"Overbought (>70): {(df['RSI'] > 70).sum():,} days ({(df['RSI'] > 70).mean()*100:.1f}%)\n"
rsi_stats += f"Oversold (<30): {(df['RSI'] < 30).sum():,} days ({(df['RSI'] < 30).mean()*100:.1f}%)"
ax1.text(0.02, 0.95, rsi_stats, transform=ax1.transAxes, verticalalignment='top',
         fontsize=9, bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# 2. MACD with Signal Line and Histogram
ax2 = axes[1]
ax2.plot(df.index, df['MACD'], linewidth=1.5, color='blue', label='MACD')
ax2.plot(df.index, df['MACD_Signal'], linewidth=1.5, color='red', label='Signal Line')
# Histogram
colors = ['green' if x >= 0 else 'red' for x in (df['MACD'] - df['MACD_Signal'])]
ax2.bar(df.index, df['MACD'] - df['MACD_Signal'], color=colors, alpha=0.5, width=1, label='Histogram')
ax2.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax2.set_title('MACD (Moving Average Convergence Divergence)', fontsize=14, fontweight='bold')
ax2.set_ylabel('MACD')
ax2.legend(loc='upper left')
ax2.grid(True, alpha=0.3)

# 3. Bollinger Bands with Price
ax3 = axes[2]
ax3.plot(df.index, df['Close'], linewidth=1.5, color='black', label='Close Price', alpha=0.7)
ax3.plot(df.index, df['BB_Upper'], linewidth=1, color='red', linestyle='--', label='Upper Band (+2σ)', alpha=0.7)
ax3.plot(df.index, df['BB_Lower'], linewidth=1, color='green', linestyle='--', label='Lower Band (-2σ)', alpha=0.7)
ax3.fill_between(df.index, df['BB_Upper'], df['BB_Lower'], alpha=0.1, color='gray')
ax3.set_title('Bollinger Bands (20-day, 2σ)', fontsize=14, fontweight='bold')
ax3.set_xlabel('Date')
ax3.set_ylabel('Price ($)')
ax3.legend(loc='upper left')
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Additional technical indicator analysis
print("\n Technical Indicators Summary:")
print(f"\nRSI Analysis:")
print(f"   Current RSI: {df['RSI'].iloc[-1]:.2f}")
print(f"   Overbought periods: {(df['RSI'] > 70).sum():,} days ({(df['RSI'] > 70).mean()*100:.1f}%)")
print(f"   Oversold periods: {(df['RSI'] < 30).sum():,} days ({(df['RSI'] < 30).mean()*100:.1f}%)")

print(f"\nMACD Analysis:")
print(f"   Current MACD: {df['MACD'].iloc[-1]:.4f}")
print(f"   Current Signal: {df['MACD_Signal'].iloc[-1]:.4f}")
current_signal = "Bullish" if df['MACD'].iloc[-1] > df['MACD_Signal'].iloc[-1] else "Bearish"
print(f"   Current Signal: {current_signal}")

print(f"\nBollinger Bands:")
current_position = (df['Close'].iloc[-1] - df['BB_Lower'].iloc[-1]) / (df['BB_Upper'].iloc[-1] - df['BB_Lower'].iloc[-1])
print(f"   Current Position: {current_position:.1%} from lower band")
if current_position > 0.8:
    print("    Price near upper band (potentially overbought)")
elif current_position < 0.2:
    print("    Price near lower band (potentially oversold)")
else:
    print("    Price in neutral zone")

## Candlestick Patterns & Event Detection

In [ ]:
# Cell 10: 
# Identify candlestick patterns
def identify_doji(df):
    """Identify Doji patterns"""
    body = abs(df['Close'] - df['Open'])
    total_range = df['High'] - df['Low']
    return body < (total_range * 0.1)

def identify_hammer(df):
    """Identify Hammer patterns"""
    body = abs(df['Close'] - df['Open'])
    lower_shadow = df[['Open', 'Close']].min(axis=1) - df['Low']
    upper_shadow = df['High'] - df[['Open', 'Close']].max(axis=1)
    return (lower_shadow > body * 2) & (upper_shadow < body)

def identify_engulfing(df):
    """Identify Bullish/Bearish Engulfing patterns"""
    body_prev = abs(df['Close'].shift(1) - df['Open'].shift(1))
    body_current = abs(df['Close'] - df['Open'])
    
    bullish = (df['Close'] > df['Open']) & (df['Close'].shift(1) < df['Open'].shift(1)) & (body_current > body_prev)
    bearish = (df['Close'] < df['Open']) & (df['Close'].shift(1) > df['Open'].shift(1)) & (body_current > body_prev)
    
    return bullish, bearish

# Identify patterns
df['Doji'] = identify_doji(df)
bullish_engulf, bearish_engulf = identify_engulfing(df)
df['Hammer'] = identify_hammer(df)
df['Bullish_Engulfing'] = bullish_engulf
df['Bearish_Engulfing'] = bearish_engulf

# Plot with patterns
fig, ax = plt.subplots(figsize=(16, 8))

# Plot price
ax.plot(df.index, df['Close'], linewidth=1, color='black', alpha=0.7, label='Close Price')

# Mark patterns
doji_dates = df[df['Doji']].index
hammer_dates = df[df['Hammer']].index
bullish_dates = df[df['Bullish_Engulfing']].index
bearish_dates = df[df['Bearish_Engulfing']].index

ax.scatter(doji_dates, df.loc[doji_dates, 'Close'], color='blue', s=20, marker='o', 
           label=f'Doji ({len(doji_dates)})', alpha=0.6)
ax.scatter(hammer_dates, df.loc[hammer_dates, 'Close'], color='green', s=20, marker='^', 
           label=f'Hammer ({len(hammer_dates)})', alpha=0.6)
ax.scatter(bullish_dates, df.loc[bullish_dates, 'Close'], color='green', s=30, marker='*', 
           label=f'Bullish Engulfing ({len(bullish_dates)})', alpha=0.8)
ax.scatter(bearish_dates, df.loc[bearish_dates, 'Close'], color='red', s=30, marker='*', 
           label=f'Bearish Engulfing ({len(bearish_dates)})', alpha=0.8)

ax.set_title('Candlestick Patterns Detection (Last 5 Years)', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Price ($)')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)

# Zoom to last 5 years for better visibility
last_5_years = df.index[-int(5*252):]
ax.set_xlim(last_5_years[0], last_5_years[-1])

plt.tight_layout()
plt.show()

# Pattern performance analysis
print("\n Candlestick Pattern Performance Analysis:")
print("\nPattern Statistics (Next Day Returns):")

patterns = ['Doji', 'Hammer', 'Bullish_Engulfing', 'Bearish_Engulfing']
for pattern in patterns:
    pattern_days = df[df[pattern]]
    if len(pattern_days) > 0:
        next_day_returns = df.loc[pattern_days.index]['Return'].shift(-1).dropna()
        if len(next_day_returns) > 0:
            avg_return = next_day_returns.mean() * 100
            win_rate = (next_day_returns > 0).mean() * 100
            print(f"\n{pattern}:")
            print(f"   Occurrences: {len(pattern_days)}")
            print(f"   Avg Next Day Return: {avg_return:.2f}%")
            print(f"   Win Rate: {win_rate:.1f}%")

# Pattern profitability
print("\n Pattern Trading Strategy (Hypothetical):")
initial_capital = 10000
capital = initial_capital

for pattern in patterns:
    pattern_days = df[df[pattern]]
    if len(pattern_days) > 0:
        returns = df.loc[pattern_days.index]['Return'].shift(-1).dropna()
        strategy_return = (1 + returns).prod() - 1
        print(f"\n{pattern} Strategy:")
        print(f"   Cumulative Return: {strategy_return*100:.2f}%")
        print(f"   Final Capital: ${initial_capital * (1 + strategy_return):,.2f}")

## Stationarity & Trend Checks

In [ ]:

def calculate_hurst(ts):
    """Calculate Hurst exponent for time series"""
    ts = np.array(ts)
    lags = range(2, min(100, len(ts)//2))
    tau = [np.sqrt(np.std(ts[lag:] - ts[:-lag])) for lag in lags]
    poly = np.polyfit(np.log(lags), np.log(tau), 1)
    return poly[0] * 2.0



fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Price rolling statistics
ax1 = axes[0, 0]
rolling_mean = df['Close'].rolling(window=252).mean()
rolling_std = df['Close'].rolling(window=252).std()

ax1.plot(df.index, df['Close'], label='Close Price', alpha=0.5, linewidth=1)
ax1.plot(df.index, rolling_mean, label='Rolling Mean (252)', linewidth=2)
ax1.fill_between(df.index, rolling_mean - rolling_std,
                 rolling_mean + rolling_std,
                 alpha=0.2, label='Rolling Std')

ax1.set_title('Price Stationarity Check')
ax1.set_xlabel('Date')
ax1.set_ylabel('Price')
ax1.legend()
ax1.grid(True)

# Returns rolling statistics
ax2 = axes[0, 1]
rolling_mean_ret = df['Return'].rolling(window=252).mean()
rolling_std_ret = df['Return'].rolling(window=252).std()

ax2.plot(df.index, df['Return'], label='Returns', alpha=0.3)
ax2.plot(df.index, rolling_mean_ret, label='Rolling Mean (252)', linewidth=2)
ax2.fill_between(df.index, rolling_mean_ret - rolling_std_ret,
                 rolling_mean_ret + rolling_std_ret,
                 alpha=0.2, label='Rolling Std')

ax2.set_title('Returns Stationarity Check')
ax2.set_xlabel('Date')
ax2.set_ylabel('Returns')
ax2.legend()
ax2.grid(True)

# ACF
ax3 = axes[1, 0]
plot_acf(df['Return'].dropna(), lags=40, ax=ax3)
ax3.set_title('Autocorrelation Function (Returns)')

# PACF
ax4 = axes[1, 1]
plot_pacf(df['Return'].dropna(), lags=40, ax=ax4)
ax4.set_title('Partial Autocorrelation Function (Returns)')

plt.tight_layout()
plt.show()



# Statistical Tests

print("\nStationarity Tests")

# ADF - Price
print("\n1. Augmented Dickey-Fuller Test (Price)")
adf_price = adfuller(df['Close'].dropna(), autolag='AIC')
print(f"ADF Statistic : {adf_price[0]:.4f}")
print(f"P-value       : {adf_price[1]:.4f}")
print("Critical Values:")
for k, v in adf_price[4].items():
    print(f"   {k} : {v:.4f}")

if adf_price[1] < 0.05:
    print("Result: Price series is stationary")
else:
    print("Result: Price series is non-stationary")


# ADF - Returns
print("\n2. Augmented Dickey-Fuller Test (Returns)")
adf_ret = adfuller(df['Return'].dropna(), autolag='AIC')
print(f"ADF Statistic : {adf_ret[0]:.4f}")
print(f"P-value       : {adf_ret[1]:.4f}")
print("Critical Values:")
for k, v in adf_ret[4].items():
    print(f"   {k} : {v:.4f}")

if adf_ret[1] < 0.05:
    print("Result: Returns are stationary")
else:
    print("Result: Returns are non-stationary")


# KPSS Test
print("\n3. KPSS Test (Price)")
kpss_price = kpss(df['Close'].dropna(), regression='c', nlags='auto')
print(f"KPSS Statistic : {kpss_price[0]:.4f}")
print(f"P-value        : {kpss_price[1]:.4f}")

if kpss_price[1] > 0.05:
    print("Result: Price is stationary")
else:
    print("Result: Price has unit root (non-stationary)")



# Mean Reversion (Hurst)

print("\nMean Reversion Analysis")

hurst = calculate_hurst(df['Return'].dropna())
print(f"Hurst Exponent : {hurst:.4f}")

if hurst < 0.5:
    print("Behavior: Mean-reverting")
elif hurst > 0.5:
    print("Behavior: Trending")
else:
    print("Behavior: Random walk")


# Trend Analysis

print("\nTrend Analysis")

df['Trend_Slope'] = df['Close'].rolling(window=1260).apply(
    lambda x: stats.linregress(range(len(x)), x)[0] if len(x) > 1 else 0
)

current_trend = df['Trend_Slope'].iloc[-1]
print(f"Current 5-year Trend Slope : {current_trend:.4f}")

if current_trend > 0:
    print("Trend Direction: Upward")
else:
    print("Trend Direction: Downward")


# Trend change detection
df['Trend_Change'] = df['Trend_Slope'].diff()
threshold = df['Trend_Change'].std() * 2
significant_changes = df[abs(df['Trend_Change']) > threshold]

print(f"\nSignificant Trend Changes Detected : {len(significant_changes)}")

top_changes = significant_changes.nlargest(5, 'Trend_Change')

print("\nTop 5 Trend Changes:")
for date, row in top_changes.iterrows():
    print(f"{date.strftime('%Y-%m-%d')} : Change = {row['Trend_Change']:.4f}")